# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.



## 1. My lane as an ML task (type)


In [3]:
!git clone https://github.com/mahmii12/FlyRank-Assignment-.git
%cd FlyRank-Assignment-
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages to rank per reviewer session")

Cloning into 'FlyRank-Assignment-'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 118 (delta 33), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 1.85 MiB | 14.82 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/FlyRank-Assignment-/FlyRank-Assignment-
30,000 pages to rank per reviewer session



This is a scoring/ranking task. The core decision is "which pages should a reviewer look at first," not a plain yes/no answer — so the output needs to be an ordered priority score across all pages, not a single classification. Ranking is the task type that directly matches how the output will be used: a reviewer works down a list, they don't get one flag per page.

## 2. Target or proxy

In [4]:
print(df['trend_direction'].value_counts())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64



The candidate label is `is_declining_label`, defined as trend_direction == "down". This is a proxy, not an observed outcome — it comes from a threshold rule applied to trend_pct (impressions_last_30d vs impressions_prev_30d), not from a real-world result like "this page was reviewed and traffic recovered." A true observed outcome would need a before/after signal tied to an actual reviewer action, which this dataset doesn't have. I'm using the proxy because it's the closest available signal to "this page needs attention," while being honest that it measures decline, not review outcomes.

## 3. Success metric

In [5]:
# Baseline vs model, from the ML-02 pipeline results
print("Baseline Precision@50:", 0.24)
print("Random forest Precision@50:", 0.74)

Baseline Precision@50: 0.24
Random forest Precision@50: 0.74



The metric is Precision@50: of the top 50 pages the model ranks highest, what fraction are actually declining (or otherwise worth reviewing)? This fits the decision because a reviewer only has time for a short list — overall accuracy or ROC-AUC would reward the model for getting the easy, low-priority cases right too, which doesn't matter here. On the starter data, the baseline rule scores Precision@50 = 0.24; a random forest model reaches 0.74 — about 3x better at putting the right pages at the top.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [6]:

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# One row = one content page (content_id), for a given client
print(df.shape)
df[["content_id", "client_id", "impressions_90d", "trend_direction",
    "trend_pct", "days_since_last_update"]].head(10)

(30000, 44)


,content_id,client_id,impressions_90d,trend_direction,trend_pct,days_since_last_update
0,content_304f48230142,client_f369cb89fc,3803,down,-41.4,20
1,content_a1fb4e703a9e,client_4e07408562,15320,down,-57.7,25
2,content_9aa793d4d895,client_7f2253d7e2,12581,down,-60.9,20
3,content_331d6c4de07b,client_19581e27de,11751,stable,-13.8,22
4,content_d99b7a2d90ca,client_3fdba35f04,19140,down,-34.7,14
5,content_d4084a4bc775,client_f369cb89fc,3970,down,-38.9,20
6,content_9a34b442b552,client_8722616204,20,down,-92.3,20
7,content_a63219c6e95a,client_19581e27de,1724,stable,0.6,22
8,content_5e6c160719bc,client_6208ef0f77,32574,down,-58.8,20
9,content_c27558df2b0c,client_19581e27de,1240,down,-29.2,104


In [7]:
print(df[["impressions_90d","trend_pct","days_since_last_update"]].corr())

                        impressions_90d  trend_pct  days_since_last_update
impressions_90d                1.000000   0.024187                0.081597
trend_pct                      0.024187   1.000000               -0.014230
days_since_last_update         0.081597  -0.014230                1.000000


## 5. Why ML beats a fixed rule here
Whether a page deserves review depends on several signals interacting — trend direction, current impression volume, position, engagement, and freshness — and their relationships aren't fixed. A page can be declining but too low-value to bother with, or stable but under-optimized and worth expanding. A single if-statement can't weigh five-plus continuous signals against each other and produce a ranked order; a model can learn how they trade off from the data itself. The precision gap (0.24 baseline vs 0.74 model) is direct evidence a fixed rule misses real patterns.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.